
# Position-Group x Year Pillar Overview

This notebook pivots the pillar scoring pipeline to a macro view. We aggregate scouting language and pillar proportions by `(Pos_Group, year)`, weight the bins by grade and player counts, and then use light topic models on the aggregated text so we can talk about canonical documents for each group without leaning on per-player noise.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
notebook_dir = Path().resolve()
if (notebook_dir / 'data').exists():
    repo_root = notebook_dir
elif (notebook_dir.parent / 'data').exists():
    repo_root = notebook_dir.parent
else:
    raise FileNotFoundError('Cannot locate repo root with data/ folder from ' + str(notebook_dir))

scores_path = repo_root / 'data' / 'processed' / 'bin_scores_v2.csv'
enriched_path = repo_root / 'data' / 'processed' / 'draft_enriched_with_contracts.csv'

df_scores = pd.read_csv(scores_path)
df_text = pd.read_csv(enriched_path)

text_fields = ['overview', 'strengths', 'weaknesses']
df_text[text_fields] = df_text[text_fields].fillna('')
df_text['all_text'] = (
    df_text['overview'] + ' ' + df_text['strengths'] + ' ' + df_text['weaknesses']
)

df_text['all_text'] = (
    df_text['all_text']
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

df = (
    df_scores
    .merge(
        df_text[['player_name', 'year', 'position', 'all_text']],
        on=['player_name', 'year', 'position'],
        how='left',
        validate='1:1'
    )
)
print(f"Loaded {len(df)} scored players; {df['all_text'].notna().mean():.0%} have text attached.")


Loaded 7173 scored players; 100% have text attached.



## Aggregating by Position Group and Year

Each `(Pos_Group, year)` bucket is summarized by player count, average grade, and total keyword hits. We show both unweighted and grade-weighted pillar proportions so the macro signal reflects the mix of high-grade prospects that scouts favored in a given era.


In [2]:

def grade_weighted(series, weights):
    weights = weights.fillna(0).clip(lower=0)
    if weights.sum() == 0:
        return series.mean()
    return np.average(series, weights=weights)


def summarize_group(group):
    grades = group['grade']
    grade_weights = grades.fillna(0).clip(lower=0)
    base = {
        'player_count': len(group),
        'avg_grade': grades.mean(),
        'grade_weight_sum': grade_weights.sum(),
        'total_matched_sum': group['total_matched'].sum(),
    }
    for bin_name in ['physical', 'technique', 'character']:
        base[f'{bin_name}_pct_mean'] = group[f'{bin_name}_pct'].mean()
        base[f'{bin_name}_pct_grade_weighted'] = grade_weighted(
            group[f'{bin_name}_pct'], grade_weights
        )
        base[f'{bin_name}_count_sum'] = group[f'{bin_name}_count'].sum()
    return pd.Series(base)

pos_year_summary = (
    df.groupby(['Pos_Group', 'year'], dropna=False)
    .apply(summarize_group)
    .reset_index()
)
pos_year_summary.sort_values(['Pos_Group', 'year'], inplace=True)

text_by_group = (
    df[['Pos_Group', 'year', 'all_text']]
    .dropna(subset=['Pos_Group', 'year'])
    .groupby(['Pos_Group', 'year'], dropna=False)['all_text']
    .agg(lambda texts: ' '.join(t for t in texts if isinstance(t, str) and t.strip()))
    .reset_index()
)
text_by_group['doc_label'] = (
    text_by_group['Pos_Group'].astype(str) + ' ' + text_by_group['year'].astype(str)
)
text_by_group = text_by_group[text_by_group['all_text'].str.strip().astype(bool)].reset_index(drop=True)

pos_year_summary = pos_year_summary.merge(
    text_by_group[['Pos_Group', 'year', 'doc_label']],
    on=['Pos_Group', 'year'],
    how='left'
)

pos_year_summary.head()


,Pos_Group,year,player_count,avg_grade,grade_weight_sum,total_matched_sum,physical_pct_mean,physical_pct_grade_weighted,physical_count_sum,technique_pct_mean,technique_pct_grade_weighted,technique_count_sum,character_pct_mean,character_pct_grade_weighted,character_count_sum,doc_label
0,DB,2010,70.0,NaN,0.00,1309.0,0.385653,0.385653,649.0,0.265970,0.265970,445.0,0.134091,0.134091,215.0,DB 2010
1,DB,2011,67.0,6.238235,212.10,1096.0,0.285706,0.365440,417.0,0.385296,0.485675,549.0,0.090185,0.125305,130.0,DB 2011
2,DB,2012,76.0,63.271528,4555.55,898.0,0.420253,0.460638,445.0,0.300600,0.325465,340.0,0.094930,0.105760,113.0,DB 2012
3,DB,2013,79.0,66.084416,5088.50,1431.0,0.354308,0.376292,636.0,0.360462,0.384439,667.0,0.070039,0.074701,128.0,DB 2013
4,DB,2014,81.0,5.717284,463.10,1342.0,0.449759,0.449998,623.0,0.327120,0.327329,455.0,0.186078,0.186390,264.0,DB 2014


## Grade Buckets (2014-2025)
We split 2014-2025 prospects into low (< 6.3) and high (= 6.3) grade cohorts so we can compare how pillar emphasis shifts between the two groups.

In [3]:
grade_df = df[df['year'].between(2014, 2025)]
grade_df = grade_df.assign(
    grade_bucket=np.where(grade_df['grade'].fillna(0) >= 6.3, 'high', 'low')
)
bucket_summary = (
    grade_df
    .groupby(['year', 'grade_bucket'])[['physical_pct', 'technique_pct', 'character_pct']]
    .mean()
    .multiply(100)
    .reset_index()
)
print('Grade bucket summary 2014-2025 (pct):')
print(bucket_summary.to_string(index=False))


Grade bucket summary 2014-2025 (pct):
 year grade_bucket  physical_pct  technique_pct  character_pct
 2014         high     39.177941      42.573824      18.247941
 2014          low     40.650213      37.573839      16.325687
 2015         high     41.665435      47.823913      10.511087
 2015          low     34.479694      40.137611       9.826889
 2016         high     40.619483      48.886552      10.494483
 2016          low     36.081760      42.364604       9.530293
 2017         high     37.764839      49.998387      10.624516
 2017          low     36.253109      44.533810       8.568599
 2018         high     37.903898      48.183220      10.522542
 2018          low     29.704332      33.701800       7.360196
 2019         high     42.881818      47.152879       9.965758
 2019          low     40.073726      45.387429      10.765047
 2020         high     38.600476      51.496667       9.902222
 2020          low     39.301519      46.562383      11.565888
 2021         hig


## Topic Modeling on Canonical Documents

Each `(Pos_Group, year)` aggregate becomes a canonical document. LDA and LSA help us surface the vocabulary that clusters around those macro documents, so we can talk about pillar language without looking at individual players.


In [4]:

try:
    from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
    from sklearn.decomposition import LatentDirichletAllocation, TruncatedSVD
except ImportError as exc:
    print('scikit-learn is required for the macro topic models:', exc)
else:
    topic_df = text_by_group.copy()
    if topic_df.empty:
        print('No aggregated text available; skipping topic modeling.')
    else:
        count_vec = CountVectorizer(max_features=3000, stop_words='english')
        dtm = count_vec.fit_transform(topic_df['all_text'])
        lda = LatentDirichletAllocation(n_components=4, random_state=42)
        lda.fit(dtm)

        feature_names = count_vec.get_feature_names_out()

        def format_topics(model, feature_names, top_n=8, use_abs=False):
            rows = []
            for idx, comp in enumerate(model.components_):
                if use_abs:
                    sorted_idx = np.argsort(np.abs(comp))[-top_n:][::-1]
                else:
                    sorted_idx = comp.argsort()[-top_n:][::-1]
                rows.append((
                    f'Topic {idx+1}',
                    ', '.join(feature_names[i] for i in sorted_idx)
                ))
            return rows

        print('LDA topics:')
        for title, terms in format_topics(lda, feature_names, top_n=8):
            print(f'  {title}: {terms}')

        doc_topic = lda.transform(dtm)
        topic_to_doc = (
            pd.DataFrame(
                doc_topic,
                columns=[f'Topic_{i+1}' for i in range(doc_topic.shape[1])]
            )
            .assign(doc_label=topic_df['doc_label'])
        )
        print('
Highest-loading documents per topic (LDA):')
        for topic in topic_to_doc.columns[:-1]:
            best_idx = topic_to_doc[topic].idxmax()
            print(f'  {topic}: {topic_to_doc.loc[best_idx, "doc_label"]}')

        tfidf_vec = TfidfVectorizer(max_features=3000, stop_words='english')
        tfidf_matrix = tfidf_vec.fit_transform(topic_df['all_text'])
        lsa = TruncatedSVD(n_components=4, random_state=42)
        lsa_doc = lsa.fit_transform(tfidf_matrix)
        tfidf_feature_names = tfidf_vec.get_feature_names_out()

        print('
LSA components:')
        for title, terms in format_topics(lsa, tfidf_feature_names, top_n=8, use_abs=True):
            print(f'  {title}: {terms}')

        print('
Highest-magnitude documents per LSA component:')
        for idx in range(lsa.components_.shape[0]):
            comp_name = f'Component_{idx+1}'
            best_idx = np.abs(lsa_doc[:, idx]).argmax()
            print(f'  {comp_name}: {topic_df.loc[best_idx, "doc_label"]}')


SyntaxError: unterminated string literal (detected at line 43) (3560874736.py, line 43)

## TF-IDF by Grade Bucket for DBs
Comparing the top TF-IDF terms for high-grade vs. low-grade defensive backs each year keeps the analysis purely corpus-driven. Run this after the grade buckets to see which words differentiate elite DBs from the rest.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

db_df = df[df['Pos_Group'] == 'DB'].copy()
db_df = db_df[db_df['year'].between(2014, 2025)]
db_df['grade_bucket'] = np.where(db_df['grade'].fillna(0) >= 6.0, 'high', 'low')

def top_terms_from_matrix(matrix, feature_names, top_n=8):
    scores = np.asarray(matrix.mean(axis=0)).ravel()
    idx = scores.argsort()[::-1][:top_n]
    return [feature_names[i] for i in idx]

for year, year_group in db_df.groupby('year'):
    texts, labels = [], []
    for bucket in ['high', 'low']:
        bucket_texts = year_group[year_group['grade_bucket'] == bucket]['all_text'].tolist()
        if not bucket_texts:
            continue
        texts.extend(bucket_texts)
        labels.extend([bucket] * len(bucket_texts))
    if not texts:
        continue
    vectorizer = TfidfVectorizer(max_features=4000, stop_words='english')
    matrix = vectorizer.fit_transform(texts)
    feature_names = vectorizer.get_feature_names_out()
    matrix_arr = matrix.toarray()
    labels = np.array(labels)

    print(f'Year {year}:')
    bucket_avgs = {}
    for bucket in ['high', 'low']:
        mask = labels == bucket
        if not mask.any():
            continue
        avg_vec = matrix_arr[mask].mean(axis=0)
        terms = top_terms_from_matrix(matrix_arr[mask], feature_names)
        bucket_avgs[bucket] = avg_vec
        print('  ' + bucket.capitalize() + f' grade ({mask.sum()} players): ' + ', '.join(terms))
    if 'high' in bucket_avgs and 'low' in bucket_avgs:
        diff = bucket_avgs['high'] - bucket_avgs['low']
        diff_idx = diff.argsort()[::-1][:8]
        diff_terms = [feature_names[i] for i in diff_idx]
        print('  Distinctive -> high-grade: ' + ', '.join(diff_terms))
    print('')


Year 2014:
  High grade (10 players): good, athletic, big, receivers, plays, quick, man, physical
  Low grade (71 players): good, ball, speed, size, teams, special, coverage, safety
  Distinctive -> high-grade: big, athletic, presence, quick, readily, match, playing, feisty

Year 2015:
  High grade (17 players): ball, coverage, speed, receivers, run, man, nfl, throws
  Low grade (59 players): coverage, play, ball, run, safety, plays, speed, receivers
  Distinctive -> high-grade: throws, nfl, inconsistent, quickly, technique, grab, outstanding, 50

Year 2016:
  High grade (20 players): ball, play, coverage, plays, run, receivers, press, speed
  Low grade (53 players): coverage, run, ball, safety, speed, receivers, plays, man
  Distinctive -> high-grade: track, yards, play, aggressive, close, starter, production, early

Year 2017:
  High grade (32 players): ball, coverage, speed, receivers, safety, play, plays, high
  Low grade (44 players): coverage, ball, run, safety, play, speed, aver